**Author:** Dorys Trujillo  
**Project:** Legal Uncertainty Index (IIJ)   
**Data Source:** Ministry of Commerce, Industry and Tourism (MinCIT)  
**Period:** 2018–2025

In this notebook, the textual corpus generated after the OCR stage is reorganized using the classification results produced by a large language model (LLM). Based on the CSV file generated during the normative extraction stage, the code identifies the regulation that each draft decree modifies or regulates and copies the corresponding text files into new folders named after that regulation. As a result, the corpus is reorganized from a year-based structure to one structured by modified regulation, facilitating subsequent analysis for the construction of legal uncertainty indicators.

In [2]:
import os
import re
import shutil
import pandas as pd
from pathlib import Path

In [14]:
# Base project directory
PROJECT_ROOT = Path(r"C:/Users/dtruj/Documentos/proyectos/legal-uncertainty-index")
# CSV generated by the LLM classification stage
CSV_ROOT = PROJECT_ROOT / "notebooks" / "mapeo_avanzado_2018_2025.csv"
# OCR corpus directory
OCR_CORPUS_DIR = PROJECT_ROOT / "data_processed" / "pdf_ocr"
# Output directory for the reorganized TXT corpus
OUTPUT_DIR = PROJECT_ROOT / "data_processed" / "txt_normative_rag"


def validate_paths(csv_path: Path, ocr_dir: Path, output_dir: Path) -> None:
    # Validate required input paths
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    if not ocr_dir.exists():
        raise FileNotFoundError(f"OCR corpus directory not found: {ocr_dir}")

    # Ensure output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)


def load_classification_csv(csv_path: Path) -> pd.DataFrame:
    # Load classification CSV
    return pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
    
    # clean column names
    df.columns = df.columns.str.strip()
    return df

def get_year_column(df: pd.DataFrame) -> str:
    # Detect the year column
    if "anio_doc" in df.columns:
        return "anio_doc"
    if "año_doc" in df.columns:
        return "año_doc"
    raise ValueError("Year column not found. Expected 'anio_doc' or 'año_doc'.")

def get_target_column(df: pd.DataFrame) -> str:
    # ensure the organization column exists
    if "carpeta_destino" not in df.columns:
        raise ValueError("The CSV must contain the column 'carpeta_destino'.")
    return "carpeta_destino"
    
def validate_required_columns(df: pd.DataFrame, year_column: str) -> None:
    # Validate required columns
    required_columns = ["archivo_pdf", year_column]
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

def clean_folder_name(name) -> str | None:
    # Normalize folder names for safe filesystem use
    if pd.isna(name):
        return None

    name = str(name)
    name = name.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r"\s+", " ", name).strip()

    # Limit folder name length to avoid Windows path issues
    name = name[:100]
    return name if name else None

def normalize_year(value) -> str | None:
    # Normalize year values such as 2018.0 to 2018
    if pd.isna(value):
        return None

    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        value = str(value).strip()
        return value if value else None

def build_txt_name(pdf_name) -> str | None:
    # Build the expected TXT filename from the PDF name
    if pd.isna(pdf_name):
        return None

    pdf_name = str(pdf_name).strip()
    if not pdf_name:
        return None

    return Path(pdf_name).with_suffix(".txt").name

def build_missing_record(
    year_doc,
    pdf_name,
    txt_name,
    expected_path
) -> dict:
    # Build a standard missing-record entry
    return {
        "anio_doc": year_doc,
        "archivo_pdf": pdf_name,
        "archivo_txt_esperado": txt_name,
        "ruta_esperada": str(expected_path)
    }

def process_row(
    archivo_pdf: str,
    anio_doc: str | None,
    nombre_carpeta: str | None,
    ocr_corpus_dir: Path,
    output_dir: Path
) -> tuple[bool, dict | None]:
    # Process a single row and return status plus missing record if needed
    archivo_txt = build_txt_name(archivo_pdf)

    if not archivo_pdf or not archivo_txt:
        return False, build_missing_record(
            anio_doc,
            archivo_pdf,
            archivo_txt,
            "Invalid or empty source filename in CSV"
        )

    if anio_doc is None:
        return False, build_missing_record(
            None,
            archivo_pdf,
            archivo_txt,
            "Year not available in CSV"
        )

    if not nombre_carpeta:
        return False, build_missing_record(
            anio_doc,
            archivo_pdf,
            archivo_txt,
            "Folder name not available in CSV"
        )

    ruta_txt_origen = ocr_corpus_dir / anio_doc / "txt" / archivo_txt
    ruta_carpeta_destino = output_dir / nombre_carpeta
    ruta_txt_destino = ruta_carpeta_destino / archivo_txt

    ruta_carpeta_destino.mkdir(parents=True, exist_ok=True)

    if ruta_txt_origen.exists() and ruta_txt_origen.is_file():
        shutil.copy2(ruta_txt_origen, ruta_txt_destino)
        return True, None

    return False, build_missing_record(
        anio_doc,
        archivo_pdf,
        archivo_txt,
        ruta_txt_origen
    )

def save_missing_report(missing_records: list[dict], output_dir: Path) -> None:
    # Save missing-file report if needed
    if not missing_records:
        return

    report_path = output_dir / "reporte_txt_faltantes.csv"
    df_missing = pd.DataFrame(missing_records)
    df_missing.to_csv(report_path, index=False, encoding="utf-8-sig", sep=";")

    print(f"Missing files report saved at: {report_path}")

def main() -> None:
    # Validate input and output paths
    validate_paths(CSV_ROOT, OCR_CORPUS_DIR, OUTPUT_DIR)

    # Load input table
    df = load_classification_csv(CSV_ROOT)

    # Resolve key columns
    year_column = get_year_column(df)
    target_column = get_target_column(df)

    # Validate mandatory fields
    validate_required_columns(df, year_column)

    copied_count = 0
    missing_records = []

    # Process each record
    for row in df.itertuples(index=False):
        archivo_pdf = getattr(row, "archivo_pdf")
        year_raw = getattr(row, year_column)
        target_raw = getattr(row, target_column)

        archivo_pdf = str(archivo_pdf).strip() if pd.notna(archivo_pdf) else ""
        anio_doc = normalize_year(year_raw)
        nombre_carpeta = clean_folder_name(target_raw)

        copied, missing_record = process_row(
            archivo_pdf=archivo_pdf,
            anio_doc=anio_doc,
            nombre_carpeta=nombre_carpeta,
            ocr_corpus_dir=OCR_CORPUS_DIR,
            output_dir=OUTPUT_DIR
        )

        if copied:
            copied_count += 1
        elif missing_record is not None:
            missing_records.append(missing_record)

    # Print process summary
    print("Process completed.")
    print(f"Files copied: {copied_count}")
    print(f"Files not found: {len(missing_records)}")

    # Save missing-file report
    save_missing_report(missing_records, OUTPUT_DIR)

if __name__ == "__main__":
    main()

Process completed.
Files copied: 329
Files not found: 0
